## Iris Failure Analysis 

### Imports

In [ ]:
import os
import time
import cv2
import pandas as pd
import iris
from tqdm.notebook import tqdm

### Setting Data Paths

In [ ]:
CASIA_ROOT = "./CASIA-Iris-Thousand/CASIA-Iris-Thousand"
IITD_ROOT = "./IITD_database"

### CASIA Image Path Function

In [ ]:
def collect_casia_paths(root):
    data = []

    for subject in os.listdir(root):
        if subject.startswith("."):
            continue

        subject_path = os.path.join(root, subject)
        if not os.path.isdir(subject_path):
            continue

        found_nested_dirs = False

        for item in os.listdir(subject_path):
            if item.startswith("."):
                continue

            item_path = os.path.join(subject_path, item)

            if os.path.isdir(item_path):
                found_nested_dirs = True
                for img in os.listdir(item_path):
                    if img.startswith("."):
                        continue
                    if img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                        data.append({
                            "subject_id": str(subject),
                            "eye": str(item),
                            "image_path": os.path.join(item_path, img),
                            "dataset": "CASIA"})

        if not found_nested_dirs:
            for img in os.listdir(subject_path):
                if img.startswith("."):
                    continue
                if img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    filename = img.lower()

                    if "_l" in filename or "left" in filename:
                        eye = "L"
                    elif "_r" in filename or "right" in filename:
                        eye = "R"
                    else:
                        eye = "unknown"

                    data.append({
                        "subject_id": str(subject),
                        "eye": eye,
                        "image_path": os.path.join(subject_path, img),
                        "dataset": "CASIA"})

    return pd.DataFrame(data)

### IITD Image Path Function

In [ ]:
def collect_iitd_paths(root):
    data = []

    for subject in os.listdir(root):
        if subject.startswith("."):
            continue

        subject_path = os.path.join(root, subject)
        if not os.path.isdir(subject_path):
            continue

        for img in os.listdir(subject_path):
            if img.startswith("."):
                continue
            if img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                filename = img.lower()

                if "_l" in filename or "left" in filename:
                    eye = "L"
                elif "_r" in filename or "right" in filename:
                    eye = "R"
                else:
                    eye = "unknown"

                data.append({
                    "subject_id": str(subject),
                    "eye": eye,
                    "image_path": os.path.join(subject_path, img),
                    "dataset": "IITD"})

    return pd.DataFrame(data)

### Load the Datasets

In [ ]:
casia_df = collect_casia_paths(CASIA_ROOT)
iitd_df = collect_iitd_paths(IITD_ROOT)

print("CASIA images:", len(casia_df))
print("IITD images:", len(iitd_df))

### Iris Pipeline

In [ ]:
pipeline = iris.IRISPipeline()

### Encoding

In [ ]:
def encode_dataset(df, pipeline, limit=None):
    records = []
    failures = []

    if limit is not None:
        df = df.head(limit).copy()

    total_images = len(df)
    start_total = time.time()

    for row in tqdm(df.itertuples(index=False), total=total_images, desc="Encoding images"):
        try:
            img = cv2.imread(row.image_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                raise ValueError("Could not read image")

            if row.eye == "L":
                eye_side = "left"
            elif row.eye == "R":
                eye_side = "right"
            else:
                eye_side = "left"

            start_img = time.time()

            output = pipeline(
                iris.IRImage(
                    img_data=img,
                    image_id=os.path.basename(row.image_path),
                    eye_side=eye_side))

            runtime = time.time() - start_img

            if output.get("error") is None:
                records.append({
                    "subject_id": row.subject_id,
                    "eye": row.eye,
                    "image_path": row.image_path,
                    "dataset": row.dataset,
                    "template": output["iris_template"],
                    "runtime": runtime})
            else:
                failures.append({
                    "subject_id": row.subject_id,
                    "eye": row.eye,
                    "image_path": row.image_path,
                    "dataset": row.dataset,
                    "error": str(output["error"]),
                    "runtime": runtime})

        except Exception as e:
            failures.append({
                "subject_id": row.subject_id,
                "eye": row.eye,
                "image_path": row.image_path,
                "dataset": row.dataset,
                "error": str(e),
                "runtime": None})

    total_runtime = time.time() - start_total
    print(f"Finished {total_images} images in {total_runtime:.2f} seconds")

    return pd.DataFrame(records), pd.DataFrame(failures)

### Running CASIA & IITD Through Pipeline

In [ ]:
casia_encoded, casia_failures = encode_dataset(casia_df, pipeline)
print("CASIA encoded:", len(casia_encoded))
print("CASIA failures:", len(casia_failures))

In [ ]:
iitd_encoded, iitd_failures = encode_dataset(iitd_df, pipeline)
print("IITD encoded:", len(iitd_encoded))
print("IITD failures:", len(iitd_failures))

### Create Failure Dataset

In [ ]:
failure_df = pd.concat([casia_failures, iitd_failures], ignore_index=True)

print("Total failures:", len(failure_df))
display(failure_df.head())

failure_df.to_csv("iris_failures.csv", index=False)
print("Saved failure dataset to iris_failures.csv")